In [0]:
%sql
INSERT INTO neha_catalog.silver_schema.cleanedmonthlysales
SELECT
    sale_id,
    product,
    category,
    quantity,
    price,
    sale_date,
    region,
    ingest_ts
FROM (
    SELECT
        CAST(TRIM(sale_id) AS INTEGER) AS sale_id,
        UPPER(TRIM(product)) AS product,
        UPPER(TRIM(category)) AS category,
        CAST(TRIM(quantity) AS INTEGER) AS quantity,
        CAST(TRIM(price) AS DOUBLE) AS price,
        TO_DATE(TRIM(sale_date), 'yyyy-MM-dd') AS sale_date,
        UPPER(TRIM(region)) AS region,
        current_timestamp AS ingest_ts,
        
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(sale_id)
            ORDER BY ingest_ts DESC
        ) AS rn

    FROM neha_catalog.bronze_schema.rawmonthlysales
    
    WHERE ingest_ts > (
        SELECT COALESCE(
            MAX(ingest_ts),
            TO_TIMESTAMP('1900-01-01', 'yyyy-MM-dd')
        )
        FROM neha_catalog.silver_schema.cleanedmonthlysales
    )
) t
WHERE rn = 1
ORDER BY sale_id;


-- INSERT INTO neha_catalog.silver_schema.cleanedmonthlysales
-- SELECT DISTINCT
--   CAST(TRIM(sale_id) AS INTEGER) AS sale_id,
--   UPPER(TRIM(product)) AS product,
--   UPPER(TRIM(category)) AS category,
--   CAST(TRIM(quantity) AS INTEGER) AS quantity,
--   CAST(TRIM(price) AS DOUBLE) AS price,
--   to_date(TRIM(sale_date), 'yyyy-MM-dd') AS sale_date,
--   UPPER(TRIM(region)) AS region,
--   current_timestamp AS ingest_ts
-- FROM neha_catalog.bronze_schema.rawmonthlysales
-- WHERE ingest_ts > (
--                     SELECT Coalesce(MAX(ingest_ts), to_date('1900-01-01', 'yyyy-MM-dd'))
--                     FROM neha_catalog.silver_schema.cleanedmonthlysales
--                   )
-- ORDER BY CAST(TRIM(sale_id) AS INTEGER);